<a href="https://colab.research.google.com/github/beingAnujChaudhary/LLM-Zoomcamp/blob/main/02-vector-search/01.vectorSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/LLM-Zoomcamp.git

# Move into repository
%cd /content/LLM-Zoomcamp

# Move into Module 2 folder
%cd 02-vector-search

Mounted at /content/drive
Cloning into 'LLM-Zoomcamp'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 88 (delta 42), reused 63 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 2.92 MiB | 6.46 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/LLM-Zoomcamp
/content/LLM-Zoomcamp/02-vector-search


# 📘 The Vector Search

---

Information Retrieval (IR) is the science of searching for documents, information within documents, and metadata about documents. The goal is to match user queries with relevant information efficiently.

**The Core Problem:** Computers need to understand human language to find relevant information. This involves:
1. Representation: How to convert text to something computable
2. Querying: How to understand user intent
3. Matching: How to find relevant documents
4. Ranking: How to order results by relevance


## Part 1: The Evolution of Search & Text Representation

### 1.1 The Information Retrieval Timeline
Information Retrieval (IR) is the science of searching for documents, information within documents, and metadata about documents.
*   **Boolean Era (1960s):** Exact matching using AND/OR/NOT operators.
*   **Probabilistic/Lexical Era (1970s-2000s):** TF-IDF and BM25. Matching based on word frequency and statistical rarity.
*   **Semantic/Dense Era (2010s-Present):** Word2Vec, Transformers, and Vector Search. Matching based on geometric proximity in high-dimensional space.



### 1.2 Keyword Search: TF-IDF and BM25
#### TF-IDF (Term Frequency-Inverse Document Frequency)
TF-IDF penalizes words that appear too frequently across all documents (like "the" or "is") and boosts words that are frequent in a specific document but rare globally.
*   **Formula:** $TF\text{-}IDF(t, d) = TF(t, d) \times \log\left(\frac{N}{df(t)}\right)$
  *   $TF$: Frequency of term $t$ in document $d$.
  *   $N$: Total number of documents.
  *   $df(t)$: Number of documents containing term $t$.

#### BM25 (Best Matching 25)
BM25 is the industry-standard probabilistic refinement of TF-IDF used in Elasticsearch and Solr. It introduces **term frequency saturation** and **document length normalization**.
*   **Formula:** $Score(D,Q) = \sum_{i=1}^{n} IDF(q_i) \times \frac{f(q_i,D) \cdot (k_1 + 1)}{f(q_i,D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{avgdl})}$
  *   $k_1$: Saturation parameter (usually 1.2 - 2.0). Prevents a term appearing 100 times from scoring 100x higher than appearing 10 times.
  *   $b$: Length normalization (usually 0.75). Penalizes longer documents.

### 1.3 The Semantic Gap & The Polysemy Problem

Keyword search suffers from the **Vocabulary Mismatch Problem**.
*   **Query:** *"Can I still join the course after the start date?"*
*   **Document:** *"Is it possible to enroll late?"*
*   **Lexical Overlap:** ~10% (only "the" or "I").
*   **Semantic Overlap:** 100%.
Keyword search fails here because it lacks context, synonym handling, and paraphrase understanding.

Furthermore, older text representations (like **One-Hot Encoding** or **Word2Vec**) fail at **Polysemy** (words with multiple meanings). The word "bank" gets the exact same vector whether you mean a river bank or a financial institution.

---



In [30]:
# Cell 1: Install Dependencies
# We install sentence-transformers for embeddings, faiss-cpu for ANN search,
# and scikit-learn for traditional keyword search (TF-IDF) comparison.
!pip install -q sentence-transformers faiss-cpu scikit-learn numpy requests matplotlib seaborn

## Part 2: Deep Dive into Embeddings (Under the Hood)

**Semantic Search:** Instead of matching discrete tokens, semantic search maps text into a continuous, dense mathematical space where semantic meaning translates to geometric proximity.

An **embedding** is a mathematical representation of discrete variables (like text, images, or audio) as continuous, dense vectors (arrays of floating-point numbers) in a high-dimensional space. The fundamental axiom of embeddings is: **Semantic similarity translates to geometric proximity.**  If two texts mean the same thing, their vectors will be close together in this space.

**Why Computers Need Embeddings?**
Machine learning models cannot read raw strings (like "Hello World"). They require numerical inputs to perform calculus and linear algebra. Embeddings provide a way to feed text into neural networks while preserving the underlying meaning, enabling tasks like semantic search, clustering, and classification.


### 2.1 Tokenization: How Machines Read Text
Neural networks cannot process raw strings. Text must first be converted into numbers via a Tokenizer.
Tokenization is the first step of the pipeline. Transformers break text into subword units.

* **Subword Tokenization (WordPiece / BPE):** Modern models don't map whole words to IDs. They break words into subword units. For example, "unhappiness" might be split into `["un", "##happi", "##ness"]`. This allows the model to understand rare words or morphological variations without needing a massive vocabulary.

* **Special Tokens:** The tokenizer adds structural markers, like `[CLS]` at the start and `[SEP]` at the end of sentences, which help the model understand boundaries.

### 2.3 The Transformer & Self-Attention
How does the model know that "judge" means something different in a legal context vs. an ML context?
* **Self-Attention Mechanism**: As the text passes through the Transformer layers, every token calculates an "attention score" against every other token in the sentence.
* In "the **judge** ruled out the crime", the word "judge" attends heavily to "ruled" and "crime", pulling its vector representation toward the legal cluster.
* In "**LLM-as-a-judge** approach", it attends to "LLM" and "evaluate", pulling its representation toward the machine learning cluster.

###  2.4 Pooling Strategies: From Tokens to Sentences
A Transformer model outputs a vector for every single token. To get a single vector for the entire sentence, we must aggregate (pool) them.
* **Mean Pooling**: Averages the vectors of all tokens in the sentence. (This is the default and most common strategy for `all-MiniLM-L6-v2`).
* **CLS Pooling**: Takes the vector of the special [CLS] token at the beginning of the sequence, which is trained to act as a aggregate representation of the whole sentence.
* **Max Pooling:** Takes the maximum value across each dimension for all tokens (less common for sentence embeddings).

### 2.5 The Transformer Architecture (The Embedding Generation Pipeline)

1. **Tokenisation: (Subwords to IDs)** `Text Input → Tokenizer` (splits into subwords, adds [CLS] and [SEP] tokens).

2.  **Embedding Layer (IDs to initial vectors):** Converts token IDs to initial dense vectors + adds Positional Encoding.

3.  **Transformer Layers (Self-Attention applies context):** Multi-head **Self-Attention** allows every token to "attend" to every other token. This is how the model solves polysemy: the word "judge" attends to "court" in legal text, but "evaluate" in ML text, resulting in different vectors.

4.  **Pooling:** To get a single vector for a sentence, we average all token vectors (**Mean Pooling**) or take the special `[CLS]` token vector.

5. **Output Vector**


### 2.6 Bi-Encoders vs. Cross-Encoders

*   **Bi-Encoders (e.g., Sentence-BERT):** Encodes the query and document *separately*. This allows us to pre-compute and store all document vectors in a database. It is extremely fast but slightly less accurate.
*   **Cross-Encoders:** Process query and document *together* simultaneously passes them through the model. It outputs a single relevance score. It is highly accurate but computationally expensive.

*   *Production Pattern:* Use a Bi-Encoder to quickly retrieve the top 100 candidates from millions of documents, then use a Cross-Encoder to rerank those 100 documents for maximum precision.

In [31]:
# Cell 2: Imports & Fetch the 1350 FAQ Dataset
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss

# Fetch the DataTalks.Club FAQ dataset
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

raw_documents = []
url_prefix = "https://datatalks.club/faq"

print("Fetching documents...")
for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    raw_documents.extend(course_response.json())

print(f"✅ Fetched {len(raw_documents)} documents.")

Fetching documents...
✅ Fetched 1350 documents.


---
## Part 3: Mathematical Foundations of Vector Space

A vector is an ordered array of numbers (e.g., `[0.12, -0.34, 0.56]`).

*   **Dimensionality:** The number of components in the array (e.g., standard models use 384, 768, or 1536 dimensions).
*   **Magnitude (Norm):** The physical length or magnitude of the vector.
*   **Direction:** The orientation in the multi-dimensional space.

When we process thousands of documents, each one becomes a single point in a high-dimensional **vector space**.

* **Semantic Proximity:** Because the neural network was trained on human language, vectors with similar meanings naturally land near each other geometrically.
* **Dense Vectors:** Unlike one-hot encoding, most values in these arrays are non-zero.

### 3.1 Latent Space vs. Embedding Space
*   **Latent Space:** is the hidden internal feature space learned by a machine learning model, where meaningful patterns and abstract concepts are represented numerically.
*   **Embedding Space:** Once every sentence has become a vector, all vectors live inside one huge mathematical space. This space is called Embedding Space.

*People often use Embedding Space and Latent Space interchangeably. They are related but not identical.*

### 3.2 Similarity Metrics
To measure how "close" two vectors are, we need a distance metric.

**1. Cosine Similarity:** Measures the *angle* ($\theta$) between vectors, ignoring magnitude.
$$ \cos(\theta) = \frac{A \cdot B}{||A||_2 \times ||B||_2} $$
*Range: [-1, 1]. (1 = identical, 0 = orthogonal, -1 = opposite).*

**2. Dot Product (Inner Product):** The algebraic sum of corresponding components.
$$ A \cdot B = \sum_{i=1}^{n} A_i B_i $$

**3. Euclidean Distance (L2):** Straight-line physical distance. Sensitive to magnitude.

> 💡 **CRUCIAL TECHNICALITY: The Normalization Trick**
> Calculating the denominator in Cosine Similarity is computationally expensive. Models like `all-MiniLM-L6-v2` apply **L2-normalization** internally, scaling vector lengths to exactly `1.0`.
> When vectors are normalized, the denominator becomes $1 \times 1 = 1$.
> **Therefore, for normalized vectors, Cosine Similarity is mathematically identical to the Dot Product.** Vector databases use Dot Product indexes under the hood because it is significantly faster.

> **Why are scores rarely negative?**
> In pure mathematics, vectors can point in exact opposite directions (-1). However, in NLP embedding spaces trained with contrastive learning, the loss function pushes unrelated sentences to be orthogonal (90 degrees, score ~0). It rarely pushes them to be exact mathematical opposites (180 degrees, -1). Therefore, the entire vector space is clustered in a single "hemisphere", meaning cosine similarities for random text pairs are almost always between 0.0 and 1.0.

In [32]:
# Cell 2: Imports & Fetch the 1350 FAQ Dataset
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import faiss

# Fetch the DataTalks.Club FAQ dataset
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

raw_documents = []
url_prefix = "https://datatalks.club/faq"

print("Fetching documents...")
for course in courses_raw:
    course_url = f"{url_prefix}{course['path']}"
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    raw_documents.extend(course_response.json())

print(f"✅ Fetched {len(raw_documents)} documents.")

Fetching documents...
✅ Fetched 1350 documents.


In [34]:
# Cell 4: Contextual Awareness (Proving the solution to Polysemy)
text_legal = "The judge ruled out the possibility of crime."
text_ml    = "We are using an LLM-as-a-judge approach to evaluate LLMs."

vec_legal = model.encode(text_legal)
vec_ml    = model.encode(text_ml)

sim_score = np.dot(vec_legal, vec_ml)

print("--- Contextual Awareness (Polysemy) ---")
print(f"Similarity between Legal 'judge' and ML 'judge': {sim_score:.4f}")
print("Notice the score is relatively low (~0.1 - 0.3). The model mapped the")
print("word 'judge' to completely different regions of the vector space based on context!")

--- Contextual Awareness (Polysemy) ---
Similarity between Legal 'judge' and ML 'judge': 0.2659
Notice the score is relatively low (~0.1 - 0.3). The model mapped the
word 'judge' to completely different regions of the vector space based on context!


---
## Part 4: The Vector Search Pipeline & ANN Algorithms

### 4.1 The Two-Phase Pipeline
1.  **Offline (Indexing):** Pass all documents through an Embedding Model to convert them into vectors and store them in a Vector Database.

Text $\rightarrow$ Chunking $\rightarrow$ Embedding Model $\rightarrow$ Vector Arrays $\rightarrow$ Vector Index.

2.  **Online (Querying):** When a user asks a question, pass the query through the exact same model. The system searches the database for document vectors closest to the query vector.

Query $\rightarrow$ Embedding Model $\rightarrow$ Query Vector $\rightarrow$ ANN Search $\rightarrow$ Top-K Results.

### 4.2 Approximate Nearest Neighbors (ANN)
Brute-force search is $O(N)$. For millions of documents, production systems use ANN to achieve $O(\log N)$ speed:
*   **HNSW (Hierarchical Navigable Small World):** Builds a multi-layered graph. Top layers are sparse (massive jumps), bottom layers are dense (precise local search). *Industry standard.*
*   **IVF (Inverted File Index):** Uses K-Means clustering to divide space into "Voronoi cells". Search only occurs in the closest clusters.
*   **PQ (Product Quantization):** Compresses vectors into sub-vectors, drastically reducing RAM usage.

### 4.3 Vector Databases

* **Vector Extensions**: Tools like `pgvector` or `sqlite-vec` add vector math capabilities to existing relational databases, ensuring ACID transactions.

* **Native Vector DBs**: Platforms like `Pinecone, Milvus, Qdrant, and ChromaDB` are built from the ground up for massive-scale vector math, advanced filtering, and clustering.

| Tool | Architecture | Best For |
| :--- | :--- | :--- |
| **Minsearch** | In-Memory (Python) | Prototyping, small datasets. |
| **SQLite-vec** | Relational Extension | Keeping metadata and vectors in ACID sync. |
| **PGVector** | PostgreSQL Extension | Enterprise apps already using Postgres. |
| **Pinecone/Qdrant** | Native Vector DB | Massive scale, distributed infrastructure. |

In [35]:
# Cell 5: The Semantic Gap (TF-IDF vs Vector Search)
# Let's prove why Keyword search fails and Vector search succeeds on a small subset.

subset_docs = [
    "Can I still join the course after the start date?",
    "Is it possible to enroll late?",
    "How do I run Docker on Windows?",
    "What is the refund policy for the bootcamp?"
]

query = "Can I enroll late?"

# --- KEYWORD SEARCH (TF-IDF) ---
vectorizer = TfidfVectorizer()
doc_tfidf = vectorizer.fit_transform(subset_docs)
query_tfidf = vectorizer.transform([query])
tfidf_scores = cosine_similarity(query_tfidf, doc_tfidf)[0]

# --- VECTOR SEARCH (Semantic) ---
query_raw = model.encode([query], convert_to_numpy=True)
query_norm = normalize(query_raw, norm='l2')
doc_embeddings = normalize(model.encode(subset_docs), norm='l2')
vector_scores = np.dot(doc_embeddings, query_norm.T).flatten()

print(f"Query: '{query}'\n")
print("Document".ljust(50), "| TF-IDF (Keyword)", "| Vector (Semantic)")
print("-" * 85)
for doc, tfidf, vec in zip(subset_docs, tfidf_scores, vector_scores):
    print(f"{doc[:48].ljust(50)} | {tfidf:.4f}           | {vec:.4f}")

Query: 'Can I enroll late?'

Document                                           | TF-IDF (Keyword) | Vector (Semantic)
-------------------------------------------------------------------------------------
Can I still join the course after the start date   | 0.1875           | 0.5779
Is it possible to enroll late?                     | 0.4870           | 0.9686
How do I run Docker on Windows?                    | 0.0000           | -0.0592
What is the refund policy for the bootcamp?        | 0.0000           | 0.2278


---
## Part 5: RAG Integration & Production Best Practices

Because Vector Search is bad at exact term matching (like specific IDs), modern Retrieval-Augmented Generation (RAG) systems use Hybrid Search.
This runs both a Keyword Search and a Vector Search simultaneously. Because BM25 scores and Cosine scores are on different mathematical scales, they are merged using Reciprocal Rank Fusion (RRF), which scores based on rankings rather than raw distances.

### 5.1 Document Chunking
Embedding models perform best on small, coherent blocks of text.
*   **Recursive Character Splitting:** Splits by paragraphs $\rightarrow$ sentences $\rightarrow$ words to preserve semantic units.
*   **Overlap:** Crucial! Overlap chunks (e.g., 500 tokens with 50 tokens overlap) so context at boundaries isn't severed.

### 5.2 Hybrid Search & Reciprocal Rank Fusion (RRF)
Vector search fails at exact keyword matching (e.g., error codes like `ERR-404`). **Hybrid Search** runs BM25 and Vector search simultaneously.
Because BM25 and Cosine scores are on different scales, we use **RRF**, which ignores raw scores and only looks at rank:
$$ RRF\_score(d) = \sum_{r \in R} \frac{1}{k + rank_r(d)} $$

*(where $k$ is a constant, usually 60, to prevent top ranks from dominating).*

### 5.3 Quantization
Storing millions of 384-dimensional Float32 vectors requires massive RAM. **Scalar Quantization** converts 32-bit floats to 8-bit integers, reducing memory footprint by 4x with only a ~1% drop in accuracy.

### 5.4 Evaluation Metrics
*   **Precision@K:** Out of the top K retrieved, how many are actually relevant?
*   **Recall@K:** Out of all relevant documents in the DB, how many were found in the top K?
*   **NDCG (Normalized Discounted Cumulative Gain):** Measures ranking quality (penalizes relevant documents appearing lower in the list).

### Production Best Practices
* **Start Simple**: Never start with vector search. Start with basic text search and upgrade to hybrid/vector once the operational complexity is justified.

* **Chunking**: LLMs have context limits. Overlap chunks (e.g., 500 tokens with 50 tokens overlap) so context isn't lost at the boundaries.

*   **ONNX:** Converts PyTorch models to an optimized graph format for blazing-fast CPU inference.

* **Quantization**: Compress high-dimensional vectors (e.g., 32-bit floats down to 8-bit integers) to reduce memory footprints by 4x to 32x with only a 1-2% drop in accuracy.

* **Batch Processing:** Encode texts in batches (e.g., 64 at a time) rather than loops.

* **Caching:** Pre-compute and store vectors to avoid redundant API or GPU costs.


In [36]:
# Cell 6: Batch Encoding the 1350 FAQ Dataset
# In production, we don't encode one by one. We use batch processing.

def prepare_text(doc):
    return f"Course: {doc.get('course', '')}\nQuestion: {doc.get('question', '')}\nAnswer: {doc.get('answer', '')}"

document_texts = [prepare_text(doc) for doc in raw_documents]

print("Encoding 1350 documents in batches...")
raw_embeddings = model.encode(
    document_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\n✅ Final Embeddings Matrix Shape: {raw_embeddings.shape}")
normalized_embeddings = normalize(raw_embeddings, norm='l2')
print("✅ All vectors successfully L2-normalized.")

Encoding 1350 documents in batches...


Batches:   0%|          | 0/22 [00:00<?, ?it/s]


✅ Final Embeddings Matrix Shape: (1350, 384)
✅ All vectors successfully L2-normalized.


In [ ]:
# Cell 7: Visualizing Semantic Similarity (Heatmap)
# Proving that documents about the same topic cluster together.

subset_indices = [0, 1, 2, 3, 4, 50, 51, 52, 100, 101]
subset_texts = [raw_documents[i]['question'][:40] for i in subset_indices]
subset_embeddings = normalized_embeddings[subset_indices]

# Calculate pairwise cosine similarity (matrix multiplication for normalized vectors)
similarity_matrix = np.dot(subset_embeddings, subset_embeddings.T)

plt.figure(figsize=(10, 8))
sns.heatmap(
    similarity_matrix,
    xticklabels=subset_texts,
    yticklabels=subset_texts,
    cmap='YlGnBu',
    annot=True,
    fmt=".2f",
    cbar_kws={'label': 'Cosine Similarity'}
)
plt.title("Pairwise Semantic Similarity of 10 FAQ Questions", fontsize=14, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

---
## Part 6: Interview Questions & Cheat Sheet

### 🎤 Top 5 Interview Questions
1.  **Q: Why do vector databases prefer Dot Product over Cosine Similarity?**
    *   *A:* Because vectors are L2-normalized. Dot Product is mathematically identical to Cosine Similarity but requires significantly fewer CPU/GPU cycles.
2.  **Q: Explain how HNSW works.**
    *   *A:* HNSW builds a multi-layered graph. Top layers are sparse (fast jumps across space), bottom layers are dense (precise local neighbor traversal).
3.  **Q: What is the Curse of Dimensionality?**
    *   *A:* As dimensions increase, data becomes sparse, and the difference between "nearest" and "farthest" Euclidean distances approaches zero. This is why Cosine Similarity (angle) is preferred over Euclidean (distance).
4.  **Q: Why use Hybrid Search?**
    *   *A:* Vector search struggles with exact matches (IDs, SKUs). Keyword search (BM25) excels at exact matches but fails on synonyms. Hybrid covers both.
5.  **Q: Explain Bi-Encoders vs. Cross-Encoders.**
    *   *A:* Bi-Encoders encode separately (fast, allows pre-computation). Cross-Encoders encode together (highly accurate, but slow). Use Bi-Encoder for retrieval, Cross-Encoder for reranking.

6.  **Q: What is Reciprocal Rank Fusion (RRF)?**
    *   *A:* RRF is an algorithm used to merge results from multiple search retrievers (like BM25 and Vector) that have incomparable score scales. It calculates a new score based purely on the document's rank in each list: $1 / (k + rank)$.

7.  **Q: How are these models actually trained? (Contrastive Learning)?**
    *   *A:* Raw BERT models are actually terrible at sentence similarity. To create sentence-transformers, researchers use a Siamese Network architecture and Contrastive Learning.
a. They feed pairs of sentences into the model.
b. If the sentences are similar (e.g., paraphrases), a loss function (like Multiple Negatives Ranking Loss) pulls their vectors closer together.
c. If they are dissimilar, it pushes them apart.
d. This fine-tuning step is what transforms a generic language model into a powerful semantic search engine.

### 📝 Quick Cheat Sheet
| Concept | Definition / Formula |
| :--- | :--- |
| **Embedding** | Dense array of floats representing semantic meaning. |
| **Cosine Similarity** | $\frac{A \cdot B}{||A|| ||B||}$. Measures angle. Range: [-1, 1]. |
| **Dot Product** | $A \cdot B$. Equals Cosine Sim if vectors are L2-normalized. |
| **HNSW** | Graph-based ANN. High memory, extremely fast, high recall. |
| **RRF** | $\sum \frac{1}{k + rank}$. Merges hybrid search results. |
| **Chunking** | Splitting text. Always use overlap (e.g., 10-20%) to preserve context. |



### 🛠️ Practical Exercises
1.  **Exercise 1:** Modify the  code below to use `IndexIVFFlat` in FAISS instead of `IndexFlatIP`. Measure the difference in search time for 10,000 random vectors.
2.  **Exercise 2:** Implement a basic RRF algorithm in Python that takes two lists of document IDs (one from BM25, one from Vector) and merges them.
3.  **Exercise 3:** Take a dataset of 100 FAQ questions. Embed them, and use UMAP to reduce the dimensions to 2D. Plot them using `matplotlib` to visually verify that similar questions cluster together.
